Storage Contract Pricer

In [2]:
import pandas as pd
import numpy as np


In [3]:
pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ----- ---------------------------------- 5.2/37.3 MB 32.5 MB/s eta 0:00:01
   ---------- ----------------------------- 9.4/37.3 MB 26.5 MB/s eta 0:00:02
   ----------- ---------------------------- 10.5/37.3 MB 18.8 MB/s eta 0:00:02
   ------------ --------------------------- 11.8/37.3 MB 14.7 MB/s eta 0:00:02
   ------------- -------------------------- 12.8/37.3 MB 12.7 MB/s eta 0:00:02
   -------------- ------------------------- 13.6/37.3 MB 11.5 MB/s eta 0:00:03
   ---------------- ----------------------- 14.9/37.3 MB 10.4 MB/s eta 0:00:03
   ----------------- ---------------------- 16.0/37.3 MB 9.7 MB/s eta 0:00:03
   ----------------- ---------------------- 16.8/37.3 MB 9.2 MB/s eta 0:00:03
   ------------------- -------------------- 17.8/37.3 MB 8.4 MB/s eta 0:00:03
   ------------------- -------------------- 18.6/37.3 MB 8.0 MB/s e


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from scipy.interpolate import CubicSpline

imports and data loading

In [11]:
# Load data
df = pd.read_csv("C:\\Users\\Renu Sharma\\Downloads\\Nat_Gas.csv")
df['Dates'] = pd.to_datetime(df['Dates'])
df = df.sort_values('Dates').reset_index(drop=True)

t_vals = (df['Dates'] - df['Dates'].min()).dt.days.values
p_vals = df['Prices'].values
cs = CubicSpline(t_vals, p_vals, extrapolate=True)

def estimate_price(date):
    d = pd.Timestamp(date)
    t = (d - df['Dates'].min()).days
    return float(cs(t))

C:\Users\Renu Sharma\AppData\Local\Temp\ipykernel_11884\3657549011.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


The full function definition:

In [12]:
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    injection_rate,
    withdrawal_rate,
    max_volume,
    storage_cost_per_month
):
    injection_dates  = [pd.Timestamp(d) for d in injection_dates]
    withdrawal_dates = [pd.Timestamp(d) for d in withdrawal_dates]

    purchase_cost = 0.0
    volume_stored = 0.0
    for d in injection_dates:
        price  = estimate_price(d)
        volume = min(injection_rate, max_volume - volume_stored)
        purchase_cost += price * volume
        volume_stored += volume
        print(f"BUY  {volume:,.0f} MMBtu @ ${price:.2f} on {d.date()}")

    revenue = 0.0
    volume_withdrawn = 0.0
    for d in sorted(withdrawal_dates):
        price  = estimate_price(d)
        volume = min(withdrawal_rate, volume_stored - volume_withdrawn)
        revenue += price * volume
        volume_withdrawn += volume
        print(f"SELL {volume:,.0f} MMBtu @ ${price:.2f} on {d.date()}")

    all_events = (
        [(d, +injection_rate)  for d in injection_dates] +
        [(d, -withdrawal_rate) for d in withdrawal_dates]
    )
    all_events.sort(key=lambda x: x[0])

    storage_cost   = 0.0
    current_volume = 0.0
    for i, (event_date, delta) in enumerate(all_events):
        if i > 0:
            prev_date     = all_events[i-1][0]
            months        = (event_date - prev_date).days / 30.44
            storage_cost += storage_cost_per_month * current_volume * months
        current_volume = max(0, min(current_volume + delta, max_volume))

    contract_value = revenue - purchase_cost - storage_cost

    print(f"\nPurchase Cost  : -${purchase_cost:,.2f}")
    print(f"Revenue        : +${revenue:,.2f}")
    print(f"Storage Cost   : -${storage_cost:,.2f}")
    print(f"Contract Value :  ${contract_value:,.2f}")

    return round(contract_value, 2)

Run the test

In [13]:
price_storage_contract(
    injection_dates        = ['2024-04-30', '2024-05-31'],
    withdrawal_dates       = ['2024-11-30', '2024-12-31'],
    injection_rate         = 500_000,
    withdrawal_rate        = 500_000,
    max_volume             = 1_000_000,
    storage_cost_per_month = 0.05
)

BUY  500,000 MMBtu @ $12.10 on 2024-04-30
BUY  500,000 MMBtu @ $11.40 on 2024-05-31
SELL 500,000 MMBtu @ $16.54 on 2024-11-30
SELL 500,000 MMBtu @ $22.58 on 2024-12-31

Purchase Cost  : -$11,750,000.00
Revenue        : +$19,559,919.51
Storage Cost   : -$351,511.17
Contract Value :  $7,458,408.34


7458408.34